# Prithvi WxC 2.3B — Run on Colab GPU
### ExtremeCast · subseasonal heatwave/coldwave forecasting

This notebook runs the **real** NASA/IBM **Prithvi WxC 2.3B** weather foundation model on a Colab GPU.
Code mirrors the official `NASA-IMPACT/Prithvi-WxC` inference example.

**Before running:** `Runtime → Change runtime type → GPU`. Pick the biggest GPU you can:
- **A100 (40 GB)** — comfortable (Colab Pro).
- **L4 (24 GB)** — works.
- **T4 (16 GB)** — works only with gradient checkpointing enabled (Cell "Memory-saver", see notes).
- **CPU** — will NOT work; the model is CUDA-only.

**Honest scope:** this notebook proves Prithvi **runs and produces a real global forecast** (zero-shot),
and gives a *starting scaffold* for fine-tuning it to the ExtremeCast heatwave task. Training it "very well"
over a long record is a larger, storage-heavy job that belongs on HPC/cloud with persistent disk — noted at the end.

## 1 · Check the GPU

In [ ]:
import torch
print("torch:", torch.__version__)
if not torch.cuda.is_available():
    raise SystemError("No GPU! Runtime -> Change runtime type -> GPU, then rerun.")
name = torch.cuda.get_device_name()
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name}  |  VRAM: {vram_gb:.1f} GB")
if vram_gb < 15:
    print("WARNING: <15 GB VRAM. Enable the memory-saver cell (gradient checkpointing) or you may OOM.")

## 2 · Install Prithvi WxC
Colab already ships CUDA torch (>=2.2), so pip won't touch it.

In [ ]:
import os, subprocess, sys
if not os.path.exists("/content/Prithvi-WxC"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/NASA-IMPACT/Prithvi-WxC.git",
                    "/content/Prithvi-WxC"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "/content/Prithvi-WxC", "huggingface_hub"], check=True)
sys.path.insert(0, "/content/Prithvi-WxC")
print("PrithviWxC installed.")

## 3 · Variables, levels, task settings
(Exact MERRA-2 variable/level set the pretrained model expects.)

In [ ]:
from pathlib import Path
import random, numpy as np, torch

random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.benchmark = True
device = torch.device("cuda")

surface_vars = ["EFLUX","GWETROOT","HFLUX","LAI","LWGAB","LWGEM","LWTUP","PS","QV2M",
                "SLP","SWGNT","SWTNT","T2M","TQI","TQL","TQV","TS","U10M","V10M","Z0M"]
static_surface_vars = ["FRACI","FRLAND","FROCEAN","PHIS"]
vertical_vars = ["CLOUD","H","OMEGA","PL","QI","QL","QV","T","U","V"]
levels = [34.,39.,41.,43.,44.,45.,48.,51.,53.,56.,63.,68.,71.,72.]
padding = {"level":[0,0], "lat":[0,-1], "lon":[0,0]}

lead_times = [18]     # forecast lead in hours (change to set the task)
input_times = [-6]    # input timestep spacing in hours
positional_encoding = "fourier"

variable_names = surface_vars + [f"{v}_level_{l}" for v in vertical_vars for l in levels]
print(f"{len(surface_vars)} surface + {len(vertical_vars)}x{len(levels)} vertical = {len(variable_names)} channels")

## 4 · Download assets from Hugging Face
Sample MERRA-2 (2020-01-01), day-1 climatology, scalers, config, and the **2.3B weights (~9 GB)**. First run takes a few minutes.

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
REPO = "ibm-nasa-geospatial/Prithvi-WxC-1.0-2300M"
DATA = "/content/data"

# MERRA-2 sample inputs (1 day)
snapshot_download(repo_id=REPO, allow_patterns="merra-2/MERRA2_sfc_2020010[1].nc", local_dir=DATA)
snapshot_download(repo_id=REPO, allow_patterns="merra-2/MERRA_pres_2020010[1].nc", local_dir=DATA)
# Climatology (day-of-year 1) — Prithvi uses a climate residual
snapshot_download(repo_id=REPO, allow_patterns="climatology/climate_surface_doy00[1]*.nc", local_dir=DATA)
snapshot_download(repo_id=REPO, allow_patterns="climatology/climate_vertical_doy00[1]*.nc", local_dir=DATA)
# Scalers (mu/sigma + anomaly variance)
for f in ["musigma_surface.nc","musigma_vertical.nc","anomaly_variance_surface.nc","anomaly_variance_vertical.nc"]:
    hf_hub_download(repo_id=REPO, filename=f"climatology/{f}", local_dir=DATA)
# Model config + weights (~9 GB)
hf_hub_download(repo_id=REPO, filename="config.yaml", local_dir=DATA)
hf_hub_download(repo_id=REPO, filename="prithvi.wxc.2300m.v1.pt", local_dir=f"{DATA}/weights")
print("Downloads complete ->", DATA)

## 5 · Build the dataset + scalers

In [ ]:
from PrithviWxC.dataloaders.merra2 import (
    Merra2Dataset, input_scalers, output_scalers, static_input_scalers)

surf_dir = Path(DATA)/"merra-2"; vert_dir = Path(DATA)/"merra-2"
surf_clim = Path(DATA)/"climatology"; vert_clim = Path(DATA)/"climatology"

dataset = Merra2Dataset(
    time_range=("2020-01-01T00:00:00","2020-01-02T05:59:59"),
    lead_times=lead_times, input_times=input_times,
    data_path_surface=surf_dir, data_path_vertical=vert_dir,
    climatology_path_surface=surf_clim, climatology_path_vertical=vert_clim,
    surface_vars=surface_vars, static_surface_vars=static_surface_vars,
    vertical_vars=vertical_vars, levels=levels, positional_encoding=positional_encoding)
assert len(dataset) > 0, "No valid data found."

sms = Path(DATA)/"climatology/musigma_surface.nc"
vms = Path(DATA)/"climatology/musigma_vertical.nc"
sos = Path(DATA)/"climatology/anomaly_variance_surface.nc"
vos = Path(DATA)/"climatology/anomaly_variance_vertical.nc"
in_mu, in_sig = input_scalers(surface_vars, vertical_vars, levels, sms, vms)
output_sig = output_scalers(surface_vars, vertical_vars, levels, sos, vos)
static_mu, static_sig = static_input_scalers(sms, static_surface_vars)
print("dataset samples:", len(dataset))

## 6 · (Optional) Memory-saver for <24 GB GPUs
Enable gradient checkpointing on a T4/16 GB. Skip on A100/L4.

In [ ]:
USE_CHECKPOINTING = torch.cuda.get_device_properties(0).total_memory/1e9 < 20
ckpt_encoder = list(range(1, 25)) if USE_CHECKPOINTING else []
ckpt_decoder = list(range(1, 5)) if USE_CHECKPOINTING else []
print("gradient checkpointing:", USE_CHECKPOINTING)

## 7 · Instantiate the 2.3B model + load weights

In [ ]:
import yaml
from PrithviWxC.model import PrithviWxC

with open(f"{DATA}/config.yaml") as f:
    cfg = yaml.safe_load(f)
p = cfg["params"]

model = PrithviWxC(
    in_channels=p["in_channels"], input_size_time=p["input_size_time"],
    in_channels_static=p["in_channels_static"],
    input_scalers_mu=in_mu, input_scalers_sigma=in_sig,
    input_scalers_epsilon=p["input_scalers_epsilon"],
    static_input_scalers_mu=static_mu, static_input_scalers_sigma=static_sig,
    static_input_scalers_epsilon=p["static_input_scalers_epsilon"],
    output_scalers=output_sig**0.5,
    n_lats_px=p["n_lats_px"], n_lons_px=p["n_lons_px"],
    patch_size_px=p["patch_size_px"], mask_unit_size_px=p["mask_unit_size_px"],
    mask_ratio_inputs=0.0, mask_ratio_targets=0.0,
    embed_dim=p["embed_dim"], n_blocks_encoder=p["n_blocks_encoder"],
    n_blocks_decoder=p["n_blocks_decoder"], mlp_multiplier=p["mlp_multiplier"],
    n_heads=p["n_heads"], dropout=p["dropout"], drop_path=p["drop_path"],
    parameter_dropout=p["parameter_dropout"], residual="climate",
    masking_mode="global", encoder_shifting=True, decoder_shifting=True,
    positional_encoding=positional_encoding,
    checkpoint_encoder=ckpt_encoder, checkpoint_decoder=ckpt_decoder)

state = torch.load(f"{DATA}/weights/prithvi.wxc.2300m.v1.pt", weights_only=False)
state = state.get("model_state", state)
model.load_state_dict(state, strict=True)
model = model.to(device)
print(f"Loaded {sum(x.numel() for x in model.parameters())/1e9:.2f}B params on {device}")

## 8 · Run a real forward pass (zero-shot forecast)
This is the proof: Prithvi produces a global forecast field.

In [ ]:
from PrithviWxC.dataloaders.merra2 import preproc

batch = preproc([next(iter(dataset))], padding)
batch = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}

model.eval()
with torch.no_grad():
    out = model(batch)
print("forecast tensor:", tuple(out.shape))  # (batch, channels, lat, lon)

## 9 · Plot the forecast (2 m temperature)

In [ ]:
import matplotlib.pyplot as plt
t2m = out[0, variable_names.index("T2M")].float().cpu().numpy()
lat = np.linspace(-90, 90, out.shape[-2]); lon = np.linspace(-180, 180, out.shape[-1])
X, Y = np.meshgrid(lon, lat)
plt.figure(figsize=(10,5))
plt.contourf(X, Y, t2m, 100)
plt.colorbar(label="T2M"); plt.gca().set_aspect("equal")
plt.title("Prithvi WxC zero-shot forecast — 2 m temperature"); plt.show()

## 10 · Fine-tuning scaffold for ExtremeCast (heatwaves)  — SCAFFOLD, not a finished result

The intended usage for a downstream task is **freeze the backbone, train a light head** (memory-cheap).
Below is the honest skeleton. It does **not** produce a trained model as-is — the TODOs mark the real work:

1. **Data:** build a `Merra2Dataset` over your full train/val period (needs the full MERRA-2 record,
   100s of GB → use HPC/cloud persistent storage, not Colab's ephemeral disk).
2. **Target:** define the heatwave target (e.g. standardized T2M anomaly / 90th-pct exceedance) from MERRA-2 or ERA5.
3. **Head:** attach a small conv/linear head on Prithvi features; freeze `model` params.
4. **Train** with an extreme-aware loss over many epochs on an A100-class GPU.

In [ ]:
# SCAFFOLD ONLY — illustrates the frozen-backbone + light-head pattern.
import torch.nn as nn

for prm in model.parameters():      # freeze the 2.3B backbone
    prm.requires_grad_(False)

class HeatwaveHead(nn.Module):
    # Maps Prithvi's output channels to a single heatwave-anomaly field.
    def __init__(self, in_ch, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, hidden, 3, padding=1), nn.GELU(),
            nn.Conv2d(hidden, 1, 1))
    def forward(self, x): return self.net(x).squeeze(1)

head = HeatwaveHead(in_ch=len(variable_names)).to(device)
opt = torch.optim.AdamW(head.parameters(), lr=3e-4)

# TODO: replace this single-sample demo with a real DataLoader + heatwave targets + many epochs.
model.eval()
with torch.no_grad():
    feats = model(batch)            # (B, C, H, W) frozen backbone features
pred = head(feats)                  # (B, H, W) heatwave-anomaly prediction
print("head output:", tuple(pred.shape), "| trainable head params:",
      sum(x.numel() for x in head.parameters()))
# loss = extreme_aware_loss(pred, target); loss.backward(); opt.step()   # <-- real training loop here

## Notes & honest limits
- **This proves Prithvi runs** and gives a real forecast + a fine-tuning starting point.
- **Full fine-tuning "very well"** needs: the full MERRA-2 record (100s of GB, persistent storage), many epochs on an
  A100-class GPU, and careful validation. Colab free/Pro sessions time out (~12–24 h) and have ephemeral disk, so use
  Colab to prototype and an HPC/cloud box for the full run.
- The ExtremeCast repo's baseline (persistence/damped + small transformer on real ERA5) is your comparison baseline —
  keep it; Prithvi must beat it to be worth reporting.
- Official reference: https://github.com/NASA-IMPACT/Prithvi-WxC · weights: https://huggingface.co/ibm-nasa-geospatial/Prithvi-WxC-1.0-2300M